# Toy Models of Feature Absorption

In [A is for Absorption: Studying Feature Splitting and Absorption in Sparse Autoencoders](https://arxiv.org/abs/2409.14507) we find evidence for a phenomenon we call "Feature absorption", where a latent which seems to track a concept has arbitrary holes in its recall. We hypothesized that this is due to the sparsity penalty incentiving the SAE to partially merge features that co-occur together to increase sparsity.

In this notebook, we set up a toy model where we can explicitly control feature representations and co-occurrence patterns, and show that feature absorption occurs in this toy setting when features co-occur.


## Install dependencies

In [ ]:
!pip install sae-lens==5.11.0 rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.1/143.1 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.0/920.0 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.6/175.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━

## Controlling feature firing and co-occurrence

Below, we set up a function `get_training_batch()` which we can use to control how many ground-truth features we have, their firing probabilities and magnitudes, and an option `modify_firing_features` callback which can be used to modify the firing features in a batch, for instance forcing a feature to fire or not fire depending on other firing features.

In [ ]:
import torch
from typing import Callable

DEFAULT_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEFAULT_D_IN = 50
DEFAULT_D_SAE = 4
DEFAULT_NUM_FEATS = 4

def get_training_batch(
    batch_size: int,
    firing_probabilities: torch.Tensor, # these are the independent probabilities of each feature firing
    std_firing_magnitudes: torch.Tensor | None = None, # If not provided, the stdev of magnitudes will be 0
    device: torch.device = DEFAULT_DEVICE,
    modify_firing_features: Callable[[torch.Tensor], torch.Tensor] | None = None,
):
    firing_features = torch.bernoulli(
        firing_probabilities.unsqueeze(0).expand(batch_size, -1).to(device)
    )
    if std_firing_magnitudes is None:
        std_firing_magnitudes = torch.zeros_like(firing_probabilities)
    if modify_firing_features is not None:
        firing_features = modify_firing_features(firing_features)
    firing_features = firing_features.to(device)
    firing_magnitude_delta = torch.normal(
        torch.zeros_like(firing_probabilities).unsqueeze(0).expand(batch_size, -1).to(device),
        std_firing_magnitudes.unsqueeze(0).expand(batch_size, -1).to(device)
    )
    return firing_features + firing_magnitude_delta

## Creating a toy model

Our toy model is simply a decoder which maps the features into a hidden dimension. The decoder is randomly initialized and we adjust the embeddings to be as orthogonal as possible from each other. If the hidden dim is less than the number of features, the features will be in superposition. To start with, we will use fully orthogonal features not in superposition. The decoder of the toy model is the "true direction" for each feature. Our hope is that a trained SAE will perfectly recover these true features directions.

In [ ]:
import torch
from torch import nn
from transformer_lens.hook_points import HookedRootModule, HookPoint
from typing import Any
from tqdm.autonotebook import tqdm

def cos_sims(mat1: torch.Tensor, mat2: torch.Tensor):
    return (mat1 / (mat1.norm(dim=0, keepdim=True))).T @ (mat2 / (mat2.norm(dim=0, keepdim=True)))

# based on https://github.com/3b1b/videos/blob/master/_2024/transformers/almost_orthogonal.py
def orthogonalize(num_vectors: int, vector_len: int) -> torch.Tensor:
    "Try to make the embeddings as orthogonal as possible, putting vectors into superposition if necessary"
    embeddings = torch.randn(num_vectors, vector_len)
    embeddings /= embeddings.norm(p=2, dim=1, keepdim=True)  # Normalize
    embeddings.requires_grad_(True)
    num_vectors = embeddings.shape[0]

    # Set up an Optimization loop to create nearly-perpendicular vectors
    optimizer = torch.optim.Adam([embeddings], lr=0.01) # type: ignore
    num_steps = 250

    losses = []

    big_id = torch.eye(num_vectors, num_vectors)

    pbar = tqdm(range(num_steps))
    for step_num in pbar:
        optimizer.zero_grad()

        dot_products = embeddings @ embeddings.T
        # Punish deviation from orthogonal
        diff = dot_products - big_id
        loss = diff.abs().pow(2).sum()
        # Extra incentive to keep rows normalized
        loss += num_vectors * diff.diag().pow(2).sum()

        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        pbar.set_description(f"loss: {loss.item():.3f}")
    emeddings = embeddings / embeddings.norm(p=2, dim=1, keepdim=True)
    embeddings.requires_grad_(False)
    return embeddings.detach().clone()


class ToyModel(HookedRootModule):
    def __init__(self, num_feats: int = DEFAULT_NUM_FEATS, hidden_dim: int = DEFAULT_D_IN):
        super().__init__()
        self.decoder = torch.nn.Linear(num_feats, hidden_dim)
        embeddings = orthogonalize(num_feats, hidden_dim)
        self.decoder.weight.data = embeddings.T
        self.setup()

    def forward(self, x: torch.Tensor, **kwargs: Any):
        x = self.decoder(x)
        return x

toy_model = ToyModel().to(DEFAULT_DEVICE)

  0%|          | 0/250 [00:00<?, ?it/s]

Let's check that our true features are orthogonal to each other

In [ ]:
import plotly.express as px

feature_cos_sims = cos_sims(toy_model.decoder.weight, toy_model.decoder.weight)

px.imshow(
    feature_cos_sims.detach().cpu().numpy(),
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="True features cosine similarities",
    height=600,
    width=600,
)

## Set up SAE training

Here, we set up a training loop to train a SAE using SAELens on our toy features and activations. This is a slightly hacky version of the main `sae_training_runner` in SAELens. We set up the SAE and the toy model to both have the same number of features.

In [ ]:
import wandb
import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch.nn import init
import math

from sae_lens import LanguageModelSAERunnerConfig
from sae_lens.training.sae_trainer import TrainingSAE, SAETrainer
from sae_lens.training.training_sae import TrainingSAEConfig
from sae_lens.training.geometric_median import compute_geometric_median


# We only need a way to get `next_batch()` from the ActivationsStore, but the real
# ActivationsStore expects tokens and a LLM rather than toy activations. For our
# purposes, this just implements the important interface to use our feature generator
class FakeActivationsStore:
    def __init__(self, model, generate_batch_fn, batch_size: int):
        self.model = model
        self.batch_size = batch_size
        self.generate_batch_fn = generate_batch_fn
        self.estimated_norm_scaling_factor = None

    def set_norm_scaling_factor_if_needed(self):
        pass

    @torch.no_grad()
    def next_batch(self):
        # the middle param is always 1 in SAELens, I think for legacy reasons
        return self.model(self.generate_batch_fn(self.batch_size)).unsqueeze(1)

# Ignore saving checkpoints, the toy models train very fast
def _save_checkpoint(trainer: SAETrainer, checkpoint_name: int | str, wandb_aliases: list[str] | None = None):
    pass

# this is copied from SAELens sae_training_runner.py. This should probably not be in the runner
def _init_sae_group_b_decs(
    sae: TrainingSAE,
    cfg: LanguageModelSAERunnerConfig,
    store: FakeActivationsStore,
) -> None:
    if cfg.b_dec_init_method == "geometric_median":
        layer_acts = store.next_batch().detach()[:, 0, :]
        # get geometric median of the activations if we're using those.
        median = compute_geometric_median(
            layer_acts,
            maxiter=100,
        ).median
        sae.initialize_b_dec_with_precalculated(median)  # type: ignore
    elif cfg.b_dec_init_method == "mean":
        layer_acts = store.next_batch().detach().cpu()[:, 0, :]
        sae.initialize_b_dec_with_mean(layer_acts)  # type: ignore

def train_toy_sae(
    toy_model: ToyModel,
    activations_batch_provider: Callable[[int], torch.Tensor],
    l1=5e-3,
    normalize_sae_decoder: bool = True,
    sae_class: type[TrainingSAE] = TrainingSAE,
    custom_init_fn: Callable[[TrainingSAE], None] | None = None,
    d_in: int = DEFAULT_D_IN,
    d_sae: int = DEFAULT_D_SAE,
) -> TrainingSAE:
    cfg = LanguageModelSAERunnerConfig(
        model_name="toy",
        hook_name="superposition_hook",
        context_size=10_000,
        d_in=d_in,
        d_sae=d_sae,
        device=str(DEFAULT_DEVICE),
        training_tokens=100_000_000,
        eval_every_n_wandb_logs=99999999999,
        l1_coefficient=l1,
        lr=3e-4,
        log_to_wandb=False,
        normalize_sae_decoder=normalize_sae_decoder,
    )
    assert cfg.d_sae is not None
    toy_model.eval()
    sae = sae_class(TrainingSAEConfig.from_dict(cfg.get_training_sae_cfg_dict()))
    # # using the default nn.Linear layer init rather than the built-in SAELens inits, this seems more consistent
    init.kaiming_uniform_(sae.W_dec, a=math.sqrt(5))
    init.kaiming_uniform_(sae.W_enc, a=math.sqrt(5))
    store = FakeActivationsStore(toy_model, activations_batch_provider, sae.cfg.context_size)
    _init_sae_group_b_decs(sae, cfg, store)
    if custom_init_fn is not None:
        custom_init_fn(sae)
    trainer = SAETrainer(
        model=toy_model,
        sae=sae,
        activation_store=store, # type: ignore
        cfg=cfg,
        save_checkpoint_fn=_save_checkpoint,
    )
    trainer.fit()
    return sae

### Plotting helpers

Some helpers for plotting feature vs latent cosine similarities and showing features with corresponding SAE activations. You can just run these.

In [ ]:
from rich.jupyter import print as rprint
from rich.table import Table
from rich.panel import Panel
from rich.text import Text

def plot_sae_feat_cos_sims(
    sae: TrainingSAE,
    model: ToyModel,
    title_suffix: str,
):
    dec_cos_sims = cos_sims(sae.W_dec.T, model.decoder.weight)
    enc_cos_sims = cos_sims(sae.W_enc, model.decoder.weight)

    fig = make_subplots(rows=1, cols=2, subplot_titles=("SAE encoder", "SAE decoder"))
    hovertemplate = "True feature: %{x}<br>SAE Latent: %{y}<br>Cosine Similarity: %{z:.3f}<extra></extra>"

    fig.add_trace(
        go.Heatmap(
            z=enc_cos_sims.detach().cpu().numpy(),
            zmin=-1,
            zmax=1,
            colorscale="RdBu",
            showscale=False,
            hovertemplate=hovertemplate,
        ),
        row=1, col=1
    )

    # Add decoder plot
    fig.add_trace(
        go.Heatmap(
            z=dec_cos_sims.detach().cpu().numpy(),
            zmin=-1,
            zmax=1,
            colorscale="RdBu",
            colorbar=dict(title="cos sim", x=1.0),
            hovertemplate=hovertemplate,
        ),
        row=1, col=2
    )

    fig.update_layout(
        height=600,
        width=1200,
        title_text=f"Cosine Similarity with True Features ({title_suffix})",
    )
    fig.update_xaxes(title_text="True feature", row=1, col=1)
    fig.update_xaxes(title_text="True feature", row=1, col=2)
    fig.update_yaxes(title_text="SAE Latent", row=1, col=1)
    fig.update_yaxes(title_text="SAE Latent", row=1, col=2)
    fig.show()

def print_sample_feats_and_acts(feats: torch.Tensor, sae: TrainingSAE, model: ToyModel):
    feat_mags = feats.float().to(DEFAULT_DEVICE)
    latent_acts = sae.encode(model(feats.float().to(DEFAULT_DEVICE)))

    table = Table(title="Sample feature values and corresponding SAE activations")

    # Add columns
    table.add_column("True features", justify="center")
    table.add_column("SAE Latent acts", justify="center")

    def style_row(row):
        text = Text()
        for val in row:
            style = "bold" if val > 1e-4 else "dim"
            text.append(f"{val:.2f}", style=style)
            text.append("  ")
        return text

    # Add rows
    for row1, row2 in zip(feat_mags, latent_acts):
        text = Text()
        table.add_row(
            style_row(row1),
            style_row(row2),
        )
    rprint(table)

## Training an SAE on fully independent features

Let's start by training a SAE to recover 4 features, all fully independent of each other. The SAE should be able to do this near perfectly.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

feat_probs = torch.tensor([0.25, 0.05, 0.05, 0.05])
generate_batch = partial(get_training_batch, firing_probabilities=feat_probs)

# NOTE: occasionaly this gets stuck in poor local minima. If this happens, try rerunning and it should converge properly.
sae = train_toy_sae(toy_model, generate_batch)

Run name: 4-L1-0.005-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


Objective value: 3824.0737:  16%|█▌        | 16/100 [00:00<00:00, 170.40it/s]
/usr/local/lib/python3.10/dist-packages/sae_lens/training/training_sae.py:472: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  out = torch.tensor(origin, dtype=self.dtype, device=self.device)
/usr/local/lib/python3.10/dist-packages/sae_lens/training/sae_trainer.py:123: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.cfg.autocast)
24400| MSE Loss 0.000 | L1 0.002: 100%|█████████▉| 99942400/100000000 [01:29<00:00, 1118631.12it/s]


In [ ]:
plot_sae_feat_cos_sims(sae, toy_model, "Fully independent features")

The SAE is able to basically perfectly recover the feature directions here. The encoder correctly filters out each feature, and the decoder perfectly matches the true feature directions.

In [ ]:
# test out feature firing patterns

test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
])

print_sample_feats_and_acts(test_feats, sae, toy_model)

Sample feature values and corresponding SAE activations
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      True features       ┃     SAE Latent acts      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1.00  0.00  0.00  0.00  │  0.00  0.00  1.00  0.00  │
│  1.00  1.00  0.00  0.00  │  0.00  0.00  1.00  1.00  │
│  0.00  0.00  1.00  0.00  │  0.00  1.00  0.00  0.00  │
│  0.00  0.00  0.00  1.00  │  1.00  0.00  0.00  0.00  │
└──────────────────────────┴──────────────────────────┘

## Feature co-occurrence leads to feature absorption

Here, feature 1 only fires if feature 0 also fires. Otherwise everything is the same as above. Feature 0's base probability is adjusted so it still fires the same proportion of times as in the above example.

You can think of this like feature 1 means "is a square", and feature 0 means "is a rectangle". All squares are rectangles, so everytime the "is a square" feature fires the "is a rectangle" feature also fires. Co-occurrence like this is extremely common in reality, as any naturally forming hierarchy will have co-occurring features like this.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

# We modify feature 1's base firing prob to be 4x the value above, since we further restrict
# it to only firing if feature 0 fires in `modify_feats()` below. This forces it to
# co-occur with feature 0, while keeping its true firing rate the same as before.
feat_probs = torch.tensor([0.25, 0.2, 0.05, 0.05])
def modify_feats(feats: torch.Tensor):
    feat_0_fires = feats[:, 0] == 1
    feats[~feat_0_fires, 1] = 0
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
)
# NOTE: occasionaly this gets stuck in poor local minima. If this happens, try rerunning and it should converge properly.
sae_abs = train_toy_sae(toy_model, generate_batch)

Run name: 4-L1-0.005-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


Objective value: 3589.2627:  14%|█▍        | 14/100 [00:00<00:00, 531.35it/s]
24400| MSE Loss 0.000 | L1 0.002: 100%|█████████▉| 99942400/100000000 [01:30<00:00, 1105840.37it/s]


In [ ]:
plot_sae_feat_cos_sims(sae_abs, toy_model, "feat 1 co-occurs w/feat 0")

Here we see feature absorption. There's a SAE latent which encodes both feature 0 and feature 1 together in the decoder, but its encoder is only matching feature 1. The latent tracking feature 0 has a decoder which perfectly reconstructs feature 0, but the encoder of this feature is excluding feature 1, making a hole in its firing pattern. Features 2 and 3 are found and reconstructed perfectly, as before.

In [ ]:
# test out feature firing patterns

test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
])

print_sample_feats_and_acts(test_feats, sae_abs, toy_model)

Sample feature values and corresponding SAE activations
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      True features       ┃     SAE Latent acts      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1.00  0.00  0.00  0.00  │  0.00  0.00  0.00  0.00  │
│  1.00  1.00  0.00  0.00  │  0.00  0.00  1.00  0.00  │
│  0.00  0.00  1.00  0.00  │  0.00  0.99  0.00  1.00  │
│  0.00  0.00  0.00  1.00  │  1.00  0.99  0.00  0.00  │
└──────────────────────────┴──────────────────────────┘

In the table above, we pass in some test features and see the corresponding SAE latent activations. When features 0 and 1 are both active together, only one latent fires, the one corresponding to feature 1, and the latent corresponding to feature 0 doesn't fire even though feature 0 is active! This is classical feature absorption - the latent corresponding to feature 1 "absorbs" feature 0.

# Magnitude variance causes partial absorption

So far each feature fires with magnitude 1.0 always. In a real model, there will be some variance in how strongly features fire. Let's update our feature generation to add some variance to feature 0, while keeping everything else the same from our feature absorption example above.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

# Like in the feature absorption example above, we force feature 1 to only fire if feature 0 also fired.
feat_probs = torch.tensor([0.25, 0.2, 0.05, 0.05])
# However, we now allow feature 0 to have some variance in its firing pattern,
# so the relative magnitudes of feature 0 and feature 1 are no longer fixed
std_firing_magnitudes = torch.tensor([0.1, 0.0, 0.0, 0.0])
def modify_feats(feats: torch.Tensor):
    feat_0_fires = feats[:, 0] == 1
    feats[~feat_0_fires, 1] = 0
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
    std_firing_magnitudes=std_firing_magnitudes,
)

# NOTE: occasionaly this gets stuck in poor local minima. If this happens, try rerunning and it should converge properly.
sae_part = train_toy_sae(toy_model, generate_batch)

Run name: 4-L1-0.005-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


Objective value: 4000.5947:  23%|██▎       | 23/100 [00:00<00:00, 545.15it/s]
24400| MSE Loss 0.000 | L1 0.003: 100%|█████████▉| 99942400/100000000 [01:29<00:00, 1121583.87it/s]


In [ ]:
plot_sae_feat_cos_sims(sae_part, toy_model, "feat 1 co-occurs w/feat 0, feat 0 magnitude varies")

This looks a lot like the original example of feature absorption, with some subtle differences. The latent tracking feature 1 still absorbs feature 0 too, but the cosine sim with feature 0 is less strong that it was in the full absorption example. Let's see what various feature firing patterns look like in SAE activations.

In [ ]:
test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0.9, 1, 0, 0],
    [0.75, 1, 0, 0],
])

print_sample_feats_and_acts(test_feats, sae_part, toy_model)

Sample feature values and corresponding SAE activations
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      True features       ┃     SAE Latent acts      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1.00  0.00  0.00  0.00  │  0.00  1.14  0.00  0.00  │
│  1.00  1.00  0.00  0.00  │  0.00  0.20  0.00  1.37  │
│  0.90  1.00  0.00  0.00  │  0.00  0.10  0.00  1.37  │
│  0.75  1.00  0.00  0.00  │  0.00  0.00  0.00  1.37  │
└──────────────────────────┴──────────────────────────┘

### Partial absorption

We see that now the SAE latent tracking feature 0 still fires when the true values of features 0 and 1 are both 1.0, but very weakly. However, if the magnitude of feature 0 drops down to 0.75, then the feature 0 latent fully turns off.

We call this phenonemon **partial absorption**. In partial absorption, there's co-occurrence between a dense and sparse feature, and the sparse feature absorbs the direction of the dense feature. However, the SAE latent tracking the dense feature still fires when both the dense and sparse feature are active, only very weakly. If the magnitude of the dense feature drops below some threshold, it stops firing entirely.

#### Why does partial absorption happen?

Feature absorption is an optimal strategy for minimizing the L1 loss and maximizing sparsity. However, when a SAE absorbs one latent into another, the absorbing latent loses the ability to modulate the magnitudes of the underlying features relative to each other. The SAE can address this by firing the latent tracking the dense feature as a "correction" to add back some of the dense feature direction into the reconstruction. Since the dense feature latent is firing weakly, it still has lower L1 loss than if the SAE fully separated out the features into their own latents.

# Imperfect co-occurrence can still lead to absorption depending on L1 penalty

Next, let's test what will happen if feature 1 is more likely to fire if feature 0 is active, but can still fire without feature 0. We set up feature 1 to fire with feature 0 95% of the time, but 5% of the time it can fire on its own.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

# Here, 95% of the time that feat 1 fires feat 0 also fires, but the other 5% of the time
# it fires feat 0 doesn't fire. We keep the overall firing rate of feat 1 at 0.05, but this
# means it's strongly correlated with feat 0 but not perfectly co-occurring
feat_probs = torch.tensor([0.25, 0.19, 0.05, 0.05])
def imperfect_correlation(feats: torch.Tensor):
    feat_0_fires = feats[:, 0] == 1
     # If feat 0 doesn't fire, feat 1 fires with this prob
     # This will keep the overall firing rate at 0.05 for feat 1
    feat_1_cond_prob = 0.0003333333
    feat_1_cond_fires = torch.bernoulli(feat_1_cond_prob * torch.ones_like(feat_0_fires)) == 1

    # if feat 0 doesn't fire, fire feat1 according to the cond prob
    feats[(~feat_0_fires * ~feat_1_cond_fires), 1] = 0
    feats[(~feat_0_fires * feat_1_cond_fires), 1] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=imperfect_correlation,
)
# NOTE: occasionaly this gets stuck in poor local minima. If this happens, try rerunning and it should converge properly.
sae_corr = train_toy_sae(toy_model, generate_batch)

Run name: 4-L1-0.005-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


Objective value: 3483.1511:  13%|█▎        | 13/100 [00:00<00:00, 555.25it/s]
23300| MSE Loss 0.000 | L1 0.002:  95%|█████████▌| 95436800/100000000 [01:28<00:04, 1071570.51it/s]

In [ ]:
plot_sae_feat_cos_sims(sae_corr, toy_model, "feat 1 partially co-occurs w/feat 0")

Looking at the cosine sim plots above, we see slight absorption happening. Let's test out some real feature activations.

In [ ]:
test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 1, 0, 0],
])

print_sample_feats_and_acts(test_feats, sae_corr, toy_model)

We still see very slight feature absorption occuring in the activation patterns. When both feature 0 and feature 1 are active, the SAE does trigger both latents but it noticely reduces the magnitude of latent 0.

Let's see what happens next if we increate the L1 coefficient

In [ ]:
# NOTE: occasionaly this gets stuck in poor local minima. If this happens, try rerunning and it should converge properly.
sae_corr_high_l1 = train_toy_sae(toy_model, generate_batch, l1=2e-2)
plot_sae_feat_cos_sims(sae_corr_high_l1, toy_model, "feat 1 partially co-occurs w/feat 0, high L1 coeff")

Run name: 4-L1-0.02-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


Objective value: 3485.5835:  13%|█▎        | 13/100 [00:00<00:00, 529.30it/s]
24400| MSE Loss 0.000 | L1 0.007: 100%|█████████▉| 99942400/100000000 [01:32<00:00, 1077947.53it/s]


Here absorption is happening much more clearly in the cosine similarity plots.

In [ ]:
test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 1, 0, 0],
])

print_sample_feats_and_acts(test_feats, sae_corr_high_l1, toy_model)

Sample feature values and corresponding SAE activations
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      True features       ┃     SAE Latent acts      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1.00  0.00  0.00  0.00  │  0.00  0.00  0.00  0.96  │
│  1.00  1.00  0.00  0.00  │  0.00  0.00  0.00  1.19  │
│  0.00  1.00  0.00  0.00  │  0.00  0.00  0.00  0.22  │
└──────────────────────────┴──────────────────────────┘

Now we see full feature absorption, where the latent tracking feature 0 fails to fire when feature 1 is active.

# Tying the encoder and decoder weights together fixes feature absorption

Looking at the encoder and decoder patterns above, it's clear that absorption requires an asymmetry between the encoder and the decoder. What if we force them to be the same?

Below, we make a new SAE class, `TiedSAE`, which forces the encoder and decoder to be the same.

In [ ]:
from torch import nuclear_norm

from sae_lens.training.training_sae import TrainingSAE, TrainingSAEConfig

class TiedSAE(TrainingSAE):
    "Hackily tie the W_enc and W_dec together"

    @property
    def W_enc(self):
        return self.W_dec.T

    def initialize_weights_basic(self):
        "Copied from SAELens, but deleting any reference to self.W_enc"
        # no config changes encoder bias init for now.
        self.b_enc = nn.Parameter(
            torch.zeros(self.cfg.d_sae, dtype=self.dtype, device=self.device)
        )

        # Start with the default init strategy:
        self.W_dec = nn.Parameter(
            torch.nn.init.kaiming_uniform_(
                torch.empty(
                    self.cfg.d_sae, self.cfg.d_in, dtype=self.dtype, device=self.device
                )
            )
        )

        # methdods which change b_dec as a function of the dataset are implemented after init.
        self.b_dec = nn.Parameter(
            torch.zeros(self.cfg.d_in, dtype=self.dtype, device=self.device)
        )

        # scaling factor for fine-tuning (not to be used in initial training)
        # TODO: Make this optional and not included with all SAEs by default (but maintain backwards compatibility)
        if self.cfg.finetuning_scaling_factor:
            self.finetuning_scaling_factor = nn.Parameter(
                torch.ones(self.cfg.d_sae, dtype=self.dtype, device=self.device)
            )

    def initialize_weights_complex(self):
        "Copied from SAELens, but deleting any reference to self.W_enc"

        if self.cfg.decoder_orthogonal_init:
            self.W_dec.data = nn.init.orthogonal_(self.W_dec.data.T).T

        elif self.cfg.decoder_heuristic_init:
            self.W_dec = nn.Parameter(
                torch.rand(
                    self.cfg.d_sae, self.cfg.d_in, dtype=self.dtype, device=self.device
                )
            )
            self.initialize_decoder_norm_constant_norm()

        if self.cfg.normalize_sae_decoder:
            with torch.no_grad():
                # Anthropic normalize this to have unit columns
                self.set_decoder_norm_to_unit_norm()

Next, let's train this on our original absorption setup

In [ ]:
# We modify feature 1's base firing prob to be 4x the value above, since we further restrict
# it to only firing if feature 0 fires in `modify_feats()` below. This forces it to
# co-occur with feature 0, while keeping its true firing rate the same as before.
feat_probs = torch.tensor([0.25, 0.2, 0.05, 0.05])
def modify_feats(feats: torch.Tensor):
    feat_0_fires = feats[:, 0] == 1
    feats[~feat_0_fires, 1] = 0
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
)
# NOTE: occasionaly this gets stuck in poor local minima. If this happens, try rerunning and it should converge properly.
sae_tied = train_toy_sae(toy_model, generate_batch, sae_class=TiedSAE)

Run name: 4-L1-0.005-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


Objective value: 3583.6138:  14%|█▍        | 14/100 [00:00<00:00, 467.41it/s]
/usr/local/lib/python3.10/dist-packages/sae_lens/training/training_sae.py:472: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

/usr/local/lib/python3.10/dist-packages/sae_lens/training/sae_trainer.py:123: FutureWarning:

`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.

24400| MSE Loss 0.000 | L1 0.002: 100%|█████████▉| 99942400/100000000 [01:32<00:00, 1084724.75it/s]


In [ ]:
plot_sae_feat_cos_sims(sae_tied, toy_model, "feat 1 co-occurs w/feat 0, tied SAE")

Without the ability to create asymmetric encoder and decoder weights, the SAE settles on the true features, despite feature co-occurrence! Hooray!

# Absorption in Superposition

So far we've had fewer features than dimensions in the residual stream. What if we have more features than residual stream dimensions? Will tying the encoder and decoder together still work?

We'll use 10 features and a 9 dimensional hidden stream, putting all 10 features into superposition.

In [ ]:
super_toy_model = ToyModel(num_feats=10, hidden_dim=9).to(DEFAULT_DEVICE)

  0%|          | 0/250 [00:00<?, ?it/s]

Let's see how similar the features in superposition are to each other. These should be mostly orthogonal, but with slight overlap.

In [ ]:
import plotly.express as px

feature_reps = super_toy_model.decoder.weight
feature_cos_sims = cos_sims(feature_reps, feature_reps)

px.imshow(
    feature_cos_sims.detach().cpu().numpy(),
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="True features cosine similarities (superposition)",
    height=600,
    width=600,
)

Now, let's set up our absorption example where feature 1 co-occurs with feature 0, but all other features fire independently with 5% probability. First let's try without tying the encoder and decoder together and verify the absorption still happens.

In [ ]:
from functools import partial

feat_probs = torch.tensor([0.25, 0.2, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])
def modify_feats(feats: torch.Tensor):
    feat_0_fires = feats[:, 0] == 1
    feats[~feat_0_fires, 1] = 0
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
)
sae_super = train_toy_sae(super_toy_model, generate_batch, d_sae=10, d_in=9, l1=3e-2)

Run name: 10-L1-0.03-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


Objective value: 5753.1841:  20%|██        | 20/100 [00:00<00:00, 508.09it/s]
/usr/local/lib/python3.10/dist-packages/sae_lens/training/training_sae.py:472: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

/usr/local/lib/python3.10/dist-packages/sae_lens/training/sae_trainer.py:123: FutureWarning:

`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.

24400| MSE Loss 0.003 | L1 0.020: 100%|█████████▉| 99942400/100000000 [01:35<00:00, 1043173.14it/s]


In [ ]:
plot_sae_feat_cos_sims(sae_super, super_toy_model, "feat 1 co-occurs w/feat 0, superposition ✨")

We still see the characteristic superposition structure in the encoder and decoder, although it's harder to see how with so many features and overlaps.

# Tying the encoder and decoder still works under full superposition

Next, let's check if tying the encoder and decoder together fixes this

In [ ]:
from functools import partial

feat_probs = torch.tensor([0.25, 0.2, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])
def modify_feats(feats: torch.Tensor):
    feat_0_fires = feats[:, 0] == 1
    feats[~feat_0_fires, 1] = 0
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
)
sae_tied_super = train_toy_sae(super_toy_model, generate_batch, sae_class=TiedSAE, d_sae=10, d_in=9, l1=3e-2)

Run name: 10-L1-0.03-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


Objective value: 5624.2148:  18%|█▊        | 18/100 [00:00<00:00, 196.79it/s]
/usr/local/lib/python3.10/dist-packages/sae_lens/training/training_sae.py:472: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

/usr/local/lib/python3.10/dist-packages/sae_lens/training/sae_trainer.py:123: FutureWarning:

`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.

24400| MSE Loss 0.005 | L1 0.023: 100%|█████████▉| 99942400/100000000 [01:35<00:00, 1045230.80it/s]


In [ ]:
plot_sae_feat_cos_sims(sae_tied_super, super_toy_model, "feat 1 co-occurs w/feat 0, tied SAE, superposition ✨")

Hooray! We're *still* able to perfectly reconstruct the true features even under superposition by tying the encoder and decoder together, although we have to increase the l1 loss coefficient significantly for this to work. We also no longer have 0 MSE loss; this could likely be fixed by using a smarter architecture like JumpReLU, or experimenting with ways of encouraging the encoder and decoder to be similar without forcing them to be identical, perhaps via an additional loss term.